# 🗞️ Tarea 1 – Parte 1: Recolección, limpieza y EDA

**Pregunta guía:** ¿Pueden convertir titulares peruanos en un clasificador de noticias?

| Campo | Valor |
|---|---|
| Estudiante | Lory Acosta |
| Opción elegida | **C – Comparada (La República + El Comercio)** |
| Fecha de descarga | se registra automáticamente en la celda siguiente (`FECHA_DESCARGA`) |
| Método | **RSS oficiales y sitemaps de noticias** declarados en el `robots.txt` de cada medio |

**URLs base**
- La República: `https://larepublica.pe` → `https://larepublica.pe/robots.txt` (lista sus feeds `/rss/*.xml`, ruta permitida con `Allow: /rss/`)
- El Comercio: `https://elcomercio.pe` → `https://elcomercio.pe/robots.txt` (lista sitemaps `/sitemap/...` y feeds `/arc/outboundfeeds/google-news-feed/...`)

**Decisión documentada:** no se hace scraping del HTML de las portadas. Se usan los canales oficiales (RSS / sitemaps de noticias) que los propios medios publican en su `robots.txt`; así se evitan bloqueos y cambios de HTML. Cada solicitud usa un User-Agent descriptivo, una pausa aleatoria de 1–2 s y **máximo 1 reintento**.

**Categorías finales (fijas):** `politica`, `economia`, `deportes`, `ciencia_tecnologia`. Lo demás se descarta y se reporta.

In [2]:
import sys, subprocess
EN_COLAB = "google.colab" in sys.modules
import plotly
if int(plotly.__version__.split(".")[0]) < 6:   # PNG estático para que los gráficos se vean en GitHub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaleido==0.2.1"])

In [3]:
import os, re, time, random, json
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from urllib.parse import urlparse
from urllib import robotparser

import requests
import pandas as pd
from bs4 import BeautifulSoup
import plotly.express as px
from IPython.display import display, Image

pd.set_option("display.max_colwidth", 120)
os.makedirs("datos/raw", exist_ok=True)

FECHA_DESCARGA = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d %H:%M %Z")
USER_AGENT = "UNAC-Tarea1-ClasificadorNoticias/1.0 (proyecto academico; requests+BeautifulSoup; 1 solicitud cada 1-2 s)"
MEDIOS = {
    "La República": "https://larepublica.pe",
    "El Comercio": "https://elcomercio.pe",
}
print("Fecha de descarga:", FECHA_DESCARGA)
print("Opción: C (comparada) | Medios:", MEDIOS)

Fecha de descarga: 2026-10-18 22:40 Hora est. Pacífico, Sudamérica
Opción: C (comparada) | Medios: {'La República': 'https://larepublica.pe', 'El Comercio': 'https://elcomercio.pe'}


## 1. Cliente HTTP respetuoso (User-Agent, pausas 1–2 s, máx. 1 reintento, robots.txt)

In [4]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "es-PE,es;q=0.9"})
ROBOTS = {}
LOG = []  # bitácora de solicitudes

def permitido(url):
    base = "{0.scheme}://{0.netloc}".format(urlparse(url))
    if base not in ROBOTS:
        rp = robotparser.RobotFileParser()
        try:
            r = SESSION.get(base + "/robots.txt", timeout=20)
            rp.parse(r.text.splitlines())
        except Exception:
            rp = None
        ROBOTS[base] = rp
    rp = ROBOTS[base]
    return True if rp is None else rp.can_fetch(USER_AGENT, url)

def get(url, intentos=2):
    """GET con pausa 1-2 s y como máximo 1 reintento (2 intentos en total)."""
    if not permitido(url):
        LOG.append((url, "bloqueado_por_robots"))
        return None
    for i in range(intentos):
        time.sleep(random.uniform(1, 2))
        try:
            r = SESSION.get(url, timeout=25)
            LOG.append((url, r.status_code))
            if r.status_code == 200:
                return r
        except requests.RequestException as e:
            LOG.append((url, type(e).__name__))
    return None

## 2. Descubrir feeds oficiales desde `robots.txt`
Se leen las líneas `Sitemap:` de cada `robots.txt` y se seleccionan solo los feeds/sitemaps de secciones relevantes (más algunas secciones *no relevantes* para documentar descartes).

In [5]:
SECCIONES_INTERES = ["politica", "economia", "deporte", "tecnologia", "ciencia",
                     "mundo", "sociedad", "home", "news", "ultimas"]

def sitemaps_de(base):
    r = get(base + "/robots.txt")
    if r is None:
        return []
    with open(f"datos/raw/robots_{urlparse(base).netloc}.txt", "w", encoding="utf-8") as f:
        f.write(r.text)
    urls = re.findall(r"(?im)^\s*sitemap:\s*(\S+)", r.text)
    return list(dict.fromkeys(urls))

FUENTES = {}
for medio, base in MEDIOS.items():
    todos = sitemaps_de(base)
    elegidos = [u for u in todos
                if any(s in u.lower().replace("í", "i").replace("ó", "o") for s in SECCIONES_INTERES)
                and "/web/" not in u]           # LR duplica cada sitemap en /sitemap/web/
    FUENTES[medio] = elegidos
    print(f"{medio}: {len(todos)} sitemaps/feeds en robots.txt → {len(elegidos)} seleccionados")

# Feeds RSS adicionales conocidos (por si robots.txt no los lista todos)
EXTRA = {
    "La República": [f"https://larepublica.pe/rss/{s}.xml" for s in
                     ["politica", "economia", "deportes", "tecnologia", "ciencia", "mundo", "sociedad"]],
    "El Comercio": [f"https://elcomercio.pe/arc/outboundfeeds/rss/category/{s}/?outputType=xml&size=100" for s in
                    ["politica", "economia", "deporte-total", "tecnologia", "ciencias", "mundo", "lima"]],
}
for medio, urls in EXTRA.items():
    FUENTES[medio] = list(dict.fromkeys(FUENTES[medio] + urls))
for medio, urls in FUENTES.items():
    print("\n", medio); [print("  ", u) for u in urls]

La República: 45 sitemaps/feeds en robots.txt → 13 seleccionados
El Comercio: 44 sitemaps/feeds en robots.txt → 23 seleccionados

 La República
   https://larepublica.pe/sitemap/sociedad.xml
   https://larepublica.pe/sitemap/politica.xml
   https://larepublica.pe/sitemap/mundo.xml
   https://larepublica.pe/sitemap/economia.xml
   https://larepublica.pe/sitemap/deportes.xml
   https://larepublica.pe/sitemap/ciencia.xml
   https://larepublica.pe/rss/sociedad.xml
   https://larepublica.pe/rss/politica.xml
   https://larepublica.pe/rss/mundo.xml
   https://larepublica.pe/rss/economia.xml
   https://larepublica.pe/rss/deportes.xml
   https://larepublica.pe/rss/ciencia.xml
   https://larepublica.pe/rss/home.xml
   https://larepublica.pe/rss/tecnologia.xml

 El Comercio
   https://elcomercio.pe/sitemap/politica/?outputType=xml
   https://elcomercio.pe/sitemap/tecnologia/?outputType=xml
   https://elcomercio.pe/sitemap/mundo/?outputType=xml
   https://elcomercio.pe/sitemap/economia/?outputType

## 3. Extracción (RSS `<item>` y sitemaps `<url>`) con respaldo crudo en `datos/raw/`

In [6]:
def limpiar_html(t):
    return re.sub(r"\s+", " ", BeautifulSoup(t or "", "html.parser").get_text(" ")).strip()

def a_fecha(txt):
    if not txt:
        return None
    txt = txt.strip()
    try:
        return parsedate_to_datetime(txt).date().isoformat()
    except Exception:
        pass
    try:
        return pd.to_datetime(txt, utc=True).date().isoformat()
    except Exception:
        return None

def seccion_de_url(url):
    partes = [p for p in urlparse(url).path.split("/") if p]
    return partes[0].lower() if partes else ""

def parsear(xml_text, medio, fuente):
    soup = BeautifulSoup(xml_text, "xml")
    filas = []
    for it in soup.find_all("item"):                       # RSS
        link = it.find("link")
        url = (link.text if link else "").strip()
        cat = it.find("category")
        sec = seccion_de_url(cat.get("domain", "")) if cat is not None and cat.get("domain") else ""
        filas.append({
            "titular": limpiar_html(it.title.text if it.title else ""),
            "bajada": limpiar_html(it.description.text if it.description else ""),
            "medio": medio,
            "seccion_original": sec or seccion_de_url(url),
            "url": url,
            "fecha": a_fecha(it.pubDate.text if it.pubDate else ""),
            "fuente": fuente,
        })
    urls_sm = soup.find_all("url")
    # sitemaps de sección pueden ser históricos y enormes: solo las MAX_POR_SITEMAP más recientes
    def _f(u):
        t = u.find("publication_date") or u.find("lastmod")
        return t.text if t else ""
    urls_sm = sorted(urls_sm, key=_f, reverse=True)[:MAX_POR_SITEMAP]
    for u in urls_sm:                                       # sitemap (con o sin extensión news)
        loc = u.find("loc")
        if loc is None:
            continue
        titulo = u.find("title") or u.find("news:title")
        fecha = u.find("publication_date") or u.find("lastmod")
        filas.append({
            "titular": limpiar_html(titulo.text if titulo else ""),
            "bajada": "",
            "medio": medio,
            "seccion_original": seccion_de_url(loc.text.strip()),
            "url": loc.text.strip(),
            "fecha": a_fecha(fecha.text if fecha else ""),
            "fuente": fuente,
        })
    return filas

MAX_POR_SITEMAP = 80
registros = []
for medio, urls in FUENTES.items():
    pendientes = list(urls)
    while pendientes:
        u = pendientes.pop(0)
        r = get(u)
        if r is None:
            print(f"  ✗ {u}")
            continue
        idx_sm = BeautifulSoup(r.content, "xml").find("sitemapindex")
        if idx_sm is not None:   # índice de sitemaps → solo los 2 primeros hijos (los más recientes)
            hijos = [l.text.strip() for l in idx_sm.find_all("loc")][:2]
            pendientes = hijos + pendientes
            print(f"  ↳ índice con {len(idx_sm.find_all('loc'))} sitemaps, se leen 2: {u}")
            continue
        nombre = re.sub(r"[^a-z0-9]+", "_", (urlparse(u).netloc + urlparse(u).path).lower()).strip("_")
        with open(f"datos/raw/{nombre}.xml", "w", encoding="utf-8") as f:
            f.write(r.text)
        filas = parsear(r.content, medio, u)
        registros += filas
        print(f"  ✓ {len(filas):4d} ítems  {u}")

crudo = pd.DataFrame(registros)
crudo.to_csv("datos/raw/noticias_crudo.csv", index=False)
print("\nTotal crudo:", crudo.shape)
crudo.groupby("medio").size()

  ✓   42 ítems  https://larepublica.pe/sitemap/sociedad.xml
  ✓   23 ítems  https://larepublica.pe/sitemap/politica.xml
  ✓   22 ítems  https://larepublica.pe/sitemap/mundo.xml
  ✓    6 ítems  https://larepublica.pe/sitemap/economia.xml
  ✓   14 ítems  https://larepublica.pe/sitemap/deportes.xml
  ✓   11 ítems  https://larepublica.pe/sitemap/ciencia.xml
  ✓   44 ítems  https://larepublica.pe/rss/sociedad.xml
  ✓   23 ítems  https://larepublica.pe/rss/politica.xml
  ✓   22 ítems  https://larepublica.pe/rss/mundo.xml
  ✓    6 ítems  https://larepublica.pe/rss/economia.xml
  ✓   14 ítems  https://larepublica.pe/rss/deportes.xml
  ✓   11 ítems  https://larepublica.pe/rss/ciencia.xml
  ✓  198 ítems  https://larepublica.pe/rss/home.xml
  ✗ https://larepublica.pe/rss/tecnologia.xml
  ↳ índice con 2 sitemaps, se leen 2: https://elcomercio.pe/sitemap/politica/?outputType=xml
  ✓   80 ítems  https://elcomercio.pe/sitemap/web/politica/?outputType=xml
  ✓   80 ítems  https://elcomercio.pe/sitemap/

medio
El Comercio     1517
La República     436
dtype: int64

### 3.1 Completar bajadas faltantes
Los sitemaps no traen bajada. Para esos casos se pide la página de la noticia **solo una vez** y se toma la meta-descripción oficial (`og:description`). Se limita a las noticias de secciones relevantes para no sobrecargar al medio.

In [7]:
RELEVANTES_RAIZ = ("politica", "economia", "deporte", "tecnolog", "ciencia")
faltan = crudo[(crudo["bajada"].fillna("") == "") &
               crudo["seccion_original"].str.startswith(RELEVANTES_RAIZ)].drop_duplicates("url")
LIMITE_ENRIQUECER = 250
print("Noticias sin bajada a completar:", min(len(faltan), LIMITE_ENRIQUECER), "de", len(faltan))

def meta_desc(url):
    r = get(url, intentos=1)
    if r is None:
        return "", ""
    s = BeautifulSoup(r.text, "html.parser")
    d = s.find("meta", property="og:description") or s.find("meta", attrs={"name": "description"})
    t = s.find("meta", property="og:title")
    return (d.get("content", "").strip() if d else ""), (t.get("content", "").strip() if t else "")

for idx, row in faltan.head(LIMITE_ENRIQUECER).iterrows():
    desc, tit = meta_desc(row["url"])
    m = crudo["url"] == row["url"]
    crudo.loc[m, "bajada"] = desc
    if not row["titular"] and tit:
        crudo.loc[m, "titular"] = tit
crudo.to_csv("datos/raw/noticias_crudo.csv", index=False)
print("Bajadas vacías restantes:", (crudo["bajada"].fillna("") == "").sum())

Noticias sin bajada a completar: 250 de 361
Bajadas vacías restantes: 515


## 4. Limpieza y normalización

In [8]:
df = crudo.copy()
n0 = len(df)
df["titular"] = df["titular"].fillna("").str.strip()
df["url"] = df["url"].fillna("").str.strip()

descartes = []
m_vacios = (df["titular"] == "") | (df["url"] == "")
descartes.append(("Titular o URL vacío", int(m_vacios.sum())))
df = df[~m_vacios]

n_antes = len(df)
df = df.drop_duplicates(subset="url", keep="first")
descartes.append(("URL duplicada (misma noticia en varios feeds)", n_antes - len(df)))

m_noticia = df["url"].str.contains(r"/(?:envivo|video|videos|galeria|fotogaleria|tag)/", regex=True)
descartes.append(("No es nota escrita (video/galería/en vivo/tag)", int(m_noticia.sum())))
df = df[~m_noticia]
print("Filas crudas:", n0, "→ tras limpieza:", len(df))

Filas crudas: 1953 → tras limpieza: 870


### 4.1 Tabla de mapeo explícito `seccion_original` → categoría

In [9]:
MAPEO = {
    # política
    "politica": "politica", "elecciones": "politica", "congreso": "politica",
    # economía
    "economia": "economia", "economia-y-finanzas": "economia", "mercados": "economia",
    "dia-1": "economia", "empresas": "economia", "finanzas": "economia",
    # deportes
    "deportes": "deportes", "deporte-total": "deportes", "futbol-peruano": "deportes",
    "futbol-internacional": "deportes", "deporte": "deportes", "dt": "deportes",
    # ciencia y tecnología
    "tecnologia": "ciencia_tecnologia", "ciencia": "ciencia_tecnologia", "ciencias": "ciencia_tecnologia",
    "tecnologia-e-innovacion": "ciencia_tecnologia", "tecnologia-y-ciencia": "ciencia_tecnologia",
    "videojuegos": "ciencia_tecnologia", "gamer": "ciencia_tecnologia",
}

def mapear(sec):
    sec = (sec or "").lower()
    if sec in MAPEO:
        return MAPEO[sec]
    for raiz, cat in [("politic", "politica"), ("econom", "economia"),
                      ("deport", "deportes"), ("tecnolog", "ciencia_tecnologia"), ("cienc", "ciencia_tecnologia")]:
        if raiz in sec:
            return cat
    return None

df["categoria"] = df["seccion_original"].map(mapear)

tabla_mapeo = (df.groupby(["medio", "seccion_original"], dropna=False)
                 .size().rename("n").reset_index())
tabla_mapeo["categoria_final"] = tabla_mapeo["seccion_original"].map(mapear).fillna("— DESCARTADA —")
tabla_mapeo = tabla_mapeo.sort_values(["categoria_final", "n"], ascending=[True, False])
display(tabla_mapeo)

,medio,seccion_original,n,categoria_final
16,El Comercio,tecnologia,76,ciencia_tecnologia
19,La República,ciencia,11,ciencia_tecnologia
1,El Comercio,deporte-total,82,deportes
23,La República,deportes,14,deportes
3,El Comercio,economia,79,economia
24,La República,economia,6,economia
12,El Comercio,politica,73,politica
5,El Comercio,elecciones,30,politica
33,La República,politica,23,politica
8,El Comercio,mag,100,— DESCARTADA —


In [10]:
m_fuera = df["categoria"].isna()
secc_desc = df.loc[m_fuera, "seccion_original"].value_counts()
descartes.append(("Sección fuera de las 4 categorías (" + ", ".join(secc_desc.index[:8]) + ")", int(m_fuera.sum())))
df = df[~m_fuera].copy()

reporte = pd.DataFrame(descartes, columns=["motivo", "noticias_descartadas"])
reporte.loc[len(reporte)] = ["TOTAL descartadas", reporte["noticias_descartadas"].sum()]
display(reporte)
print(f"Noticias crudas: {n0} | Válidas finales: {len(df)}")
print("\nSecciones descartadas (top):"); print(secc_desc.head(15).to_string())

,motivo,noticias_descartadas
0,Titular o URL vacío,203
1,URL duplicada (misma noticia en varios feeds),880
2,No es nota escrita (video/galería/en vivo/tag),0
3,"Sección fuera de las 4 categorías (mundo, mag, sociedad, opinion, lima, espectaculos, respuestas, luces)",476
4,TOTAL descartadas,1559


Noticias crudas: 1953 | Válidas finales: 394

Secciones descartadas (top):
seccion_original
mundo             101
mag               100
sociedad           43
opinion            40
lima               40
espectaculos       32
respuestas         25
luces              15
peru               11
eldominical        10
latinos-en-usa      8
ecdata              8
somos               8
cultural            5
cine-series         5


### 4.2 Verificación de URLs funcionales (muestra)
Se comprueban 15 URLs al azar con la misma política de pausas.

In [11]:
muestra = df.sample(min(15, len(df)), random_state=42)
estado = []
for u in muestra["url"]:
    time.sleep(random.uniform(1, 2))
    try:
        estado.append(SESSION.head(u, timeout=20, allow_redirects=True).status_code)
    except requests.RequestException as e:
        estado.append(type(e).__name__)
verif = pd.DataFrame({"url": muestra["url"].values, "status": estado})
display(verif)
print("URLs OK (200):", (verif["status"] == 200).sum(), "/", len(verif))

,url,status
0,https://elcomercio.pe/politica/actualidad/en-vivo-comision-del-senado-dictamen-sobre-pedido-de-facultades-del-gobier...,200
1,https://elcomercio.pe/economia/opinion/norwayperu-investment-forum-2026-reunira-a-empresas-lideres-para-impulsar-nue...,200
2,https://elcomercio.pe/economia/peru/gobierno-transfiere-mas-de-s19-millones-a-gobiernos-regionales-para-cumplir-con-...,200
3,https://elcomercio.pe/politica/el-voto-31-este-es-el-escenario-que-muestra-el-peso-de-obras-para-el-fujimorismo-en-l...,200
4,https://elcomercio.pe/elecciones/resultados-en-san-borja-de-las-elecciones-regionales-y-municipales-2026-segun-conte...,200
5,https://elcomercio.pe/economia/mercados/precio-del-dolar-en-peru-hoy-domingo-18-de-octubre-a-cuanto-esta-el-dolar-y-...,200
6,https://larepublica.pe/deportes/2026/10/17/partidos-de-hoy-sabado-17-de-octubre-horarios-y-canales-para-ver-futbol-e...,200
7,https://elcomercio.pe/economia/mundo/encargado-de-turismo-de-estados-unidos-defiende-su-liderazgo-turistico-pese-a-l...,200
8,https://elcomercio.pe/tecnologia/actualidad/netflix-despedira-a-unos-1000-empleados-el-6-de-su-plantilla-segun-medio...,200
9,https://elcomercio.pe/politica/gobierno/keiko-fujimori-sobre-delegacion-de-facultades-esperemos-que-el-debate-se-ret...,200


URLs OK (200): 15 / 15


## 5. Análisis exploratorio (Plotly Express)

In [12]:
os.makedirs("figuras", exist_ok=True)
_n = [0]
def mostrar(fig):
    """Muestra el gráfico interactivo, lo guarda en figuras/*.html y agrega una copia PNG (visible en GitHub)."""
    _n[0] += 1
    fig.write_html(f"figuras/eda_{_n[0]}.html", include_plotlyjs="cdn")
    fig.show()
    try:
        png = fig.to_image(format="png", width=950, height=480, scale=1)
        open(f"figuras/eda_{_n[0]}.png", "wb").write(png)
        display(Image(png))
    except Exception as e:
        print("(PNG estático no disponible:", type(e).__name__, ")")

df["len_titular"] = df["titular"].str.len()
orden = ["politica", "economia", "deportes", "ciencia_tecnologia"]

In [13]:
cnt = df["categoria"].value_counts().reindex(orden).fillna(0).astype(int).reset_index()
cnt.columns = ["categoria", "n"]
top, bajo = cnt.loc[cnt.n.idxmax()], cnt.loc[cnt.n.idxmin()]
ratio = top.n / max(bajo.n, 1)
titulo1 = (f"'{top.categoria}' domina con {top.n/cnt.n.sum():.0%} de las noticias; "
           f"'{bajo.categoria}' es la minoritaria ({ratio:.1f}× menos) → dataset desbalanceado, usar F1 macro")
fig1 = px.bar(cnt, x="categoria", y="n", text="n", color="categoria", title=titulo1,
              labels={"n": "Noticias", "categoria": "Categoría"})
fig1.update_layout(showlegend=False, title_font_size=14)
mostrar(fig1)

(PNG estático no disponible: RuntimeError )


In [14]:
cm = df.groupby(["medio", "categoria"]).size().reset_index(name="n")
pct = cm.pivot(index="categoria", columns="medio", values="n").fillna(0)
pct = pct / pct.sum()
dif = (pct.iloc[:, 0] - pct.iloc[:, 1]).abs() if pct.shape[1] == 2 else pd.Series([0])
cat_dif = dif.idxmax() if pct.shape[1] == 2 else "-"
tot = df["medio"].value_counts()
titulo2 = (f"{tot.index[0]} aporta {tot.iloc[0]/tot.sum():.0%} del corpus; la mayor diferencia de "
           f"mezcla editorial entre medios está en '{cat_dif}' ({dif.max():.0%} pts.)")
fig2 = px.bar(cm, x="medio", y="n", color="categoria", barmode="group", text="n", title=titulo2,
              category_orders={"categoria": orden}, labels={"n": "Noticias", "medio": "Medio"})
fig2.update_layout(title_font_size=14)
mostrar(fig2)

print("Rango de fechas:", df["fecha"].min(), "→", df["fecha"].max())

(PNG estático no disponible: RuntimeError )
Rango de fechas: 2026-10-07 → 2026-10-19


In [15]:
med = df["len_titular"].median()
p10, p90 = df["len_titular"].quantile([.1, .9])
med_cat = df.groupby("categoria")["len_titular"].median().sort_values()
titulo3 = (f"Los titulares son cortos (mediana {med:.0f} caracteres; 80% entre {p10:.0f} y {p90:.0f}): "
           f"'{med_cat.index[-1]}' titula más largo y '{med_cat.index[0]}' más corto → max_length=64 tokens basta")
fig3 = px.histogram(df, x="len_titular", nbins=30, color="medio", barmode="overlay", opacity=0.7,
                    title=titulo3, labels={"len_titular": "Longitud del titular (caracteres)"})
fig3.add_vline(x=med, line_dash="dash", annotation_text=f"mediana={med:.0f}")
fig3.update_layout(title_font_size=13)
mostrar(fig3)

(PNG estático no disponible: RuntimeError )


## 6. Validación: 5 titulares más largos y 5 más cortos

In [16]:
cols = ["len_titular", "titular", "medio", "categoria"]
print("▶ 5 titulares MÁS LARGOS"); display(df.nlargest(5, "len_titular")[cols])
print("▶ 5 titulares MÁS CORTOS"); display(df.nsmallest(5, "len_titular")[cols])
print(f"Mín={df.len_titular.min()} | Máx={df.len_titular.max()} | Mediana={med:.0f} "
      "→ coinciden con los extremos y el centro del histograma.")

▶ 5 titulares MÁS LARGOS


,len_titular,titular,medio,categoria
1185,178,"“El jugador peruano tiene un déficit grande en nutrición y táctica que resuelve a los 20 años, eso en Argentina es a...",El Comercio,deportes
1186,171,El furioso “Que se dedique a administrar” de Guede a Franco Velazco y un Quevedo que reconquistó a Comando Sur: Lo q...,El Comercio,deportes
1416,167,“Creo que puedo llegar a un top 3 sin que nadie se quede botado”: Raúl ‘Mono’ Orlandini le responde a El Comercio so...,El Comercio,deportes
501,166,"Keiko Fujimori sobre Álvaro Vargas Llosa: “Confío en que su trayectoria, experiencia y compromiso contribuirán a for...",El Comercio,politica
109,164,"En 1986, la explosión del reactor de Chernóbil obligó a que los perros fueran abandonados por sus dueños; décadas de...",La República,ciencia_tecnologia


▶ 5 titulares MÁS CORTOS


,len_titular,titular,medio,categoria
1052,22,Invocación por el Perú,El Comercio,economia
923,23,AlaCena no tiene precio,El Comercio,economia
944,24,IA y Sandbox regulatorio,El Comercio,economia
1053,41,La evolución de la confianza en los pagos,El Comercio,economia
925,43,Premio Nobel de Economía: la macro es micro,El Comercio,economia


Mín=22 | Máx=178 | Mediana=100 → coinciden con los extremos y el centro del histograma.


## 7. Exportación de `datos/noticias.csv` (7 columnas)

In [17]:
COLS = ["titular", "bajada", "medio", "seccion_original", "categoria", "url", "fecha"]
final = df[COLS].reset_index(drop=True)
final.to_csv("datos/noticias.csv", index=False, encoding="utf-8")
print("df.shape =", final.shape)
assert final.shape[1] == 7
if len(final) < 350:
    print("⚠️ Menos de 350 filas: volver a ejecutar más tarde (los feeds se renuevan) — las nuevas se suman al crudo.")
final.head()

df.shape = (394, 7)


,titular,bajada,medio,seccion_original,categoria,url,fecha
0,Carlos Infante: “La matanza de Uchuraccay es un golpe terrible a la memoria de los periodistas”,"El periodista Carlos Infante, entre la memoria personal y la memoria histórica, nos ofrece en ""Dame tu cámara carajo...",La República,politica,politica,https://larepublica.pe/politica/2026/08/15/carlos-infante-la-matanza-de-uchuraccay-es-un-golpe-terrible-a-la-memoria...,2026-10-18
1,"Jorge Nieto a Álvaro Vargas Llosa: ""Su rol de comentarista político ya acabó, debería comportarse como un embajador""",El líder de Buen Gobierno resaltó que su partido no está obstaculizando el pedido de facultades legislativas present...,La República,politica,politica,https://larepublica.pe/politica/2026/10/18/jorge-nieto-a-alvaro-vargas-llosa-su-rol-de-comentarista-politico-ya-acab...,2026-10-19
2,Comisión de Ética recomienda suspender por 120 días al diputado Julián Pérez Mallqui,"Según el documento, la conducta atribuida al diputado afectó la imagen institucional del Congreso y contravino las o...",La República,politica,politica,https://larepublica.pe/politica/2026/10/18/comision-de-etica-recomienda-suspender-por-120-dias-al-diputado-julian-pe...,2026-10-19
3,Keiko Fujimori coordinó con Delcy Rodríguez la expulsión de venezolanos indocumentados,"La mandataria informó que su Gobierno iniciará el apoyo al programa Vuelta a la Patria, iniciativa venezolana orient...",La República,politica,politica,https://larepublica.pe/politica/2026/10/18/keiko-fujimori-coordina-con-delcy-rodriguez-la-expulsion-de-venezolanos-i...,2026-10-19
4,Detención de seis policías por presunto robo reaviva debate sobre reforma de la PNP,La situación ha llevado a que la desconfianza en la PNP aumente porcentualmente a picos históricos. Experto señala q...,La República,politica,politica,https://larepublica.pe/politica/2026/10/18/detencion-de-seis-policias-por-presunto-robo-reaviva-debate-sobre-reforma...,2026-10-18


## Conclusiones de la Parte 1
- Los feeds oficiales (RSS + sitemaps de noticias declarados en `robots.txt`) permitieron obtener el corpus sin scraping agresivo ni bloqueos.
- La tabla de mapeo deja explícito qué secciones se convierten en cada una de las 4 categorías; el resto se descartó y se reportó con su motivo.
- La distribución por categoría está desbalanceada, por eso en la Parte 2 la métrica principal es **F1 macro** y el split es **estratificado**.
- Los titulares son textos cortos, lo que permite entrenar el Transformer con `max_length=64`.